In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
os.makedirs('charts', exist_ok=True)

matches = pd.read_csv('matches_clean.csv')
deliveries = pd.read_csv('deliveries_clean.csv')

print("Data loaded for Win Probability Model")

In [ ]:
# For each over we calculate the match state
# Features: runs scored, wickets lost, run rate, required run rate

# First get first innings totals per match
first_innings = deliveries[deliveries['inning'] == 1].groupby('match_id').agg(
    first_innings_total=('total_runs', 'sum')
).reset_index()

# Second innings ball by ball
second_innings = deliveries[deliveries['inning'] == 2].copy()
second_innings = second_innings.merge(first_innings, on='match_id')
second_innings = second_innings.merge(
    matches[['id', 'winner', 'team1', 'team2']],
    left_on='match_id', right_on='id'
)

print(f"Second innings deliveries: {len(second_innings)}")
print(second_innings.head())

In [ ]:
# Sort by match and ball
second_innings = second_innings.sort_values(['match_id', 'inning', 'over', 'ball'])

# Calculate cumulative runs and wickets
second_innings['cumulative_runs'] = second_innings.groupby('match_id')['total_runs'].cumsum()
second_innings['cumulative_wickets'] = second_innings.groupby('match_id')['is_wicket'].cumsum()

# Balls remaining
second_innings['ball_number'] = second_innings.groupby('match_id').cumcount() + 1
second_innings['balls_remaining'] = 120 - second_innings['ball_number']

# Runs needed
second_innings['runs_needed'] = second_innings['first_innings_total'] - second_innings['cumulative_runs'] + 1

# Current run rate
second_innings['current_rr'] = (second_innings['cumulative_runs'] / second_innings['ball_number']) * 6

# Required run rate
second_innings['required_rr'] = (second_innings['runs_needed'] / second_innings['balls_remaining'].replace(0, 1)) * 6

# Did batting team (second innings) win?
second_innings['batting_team_won'] = (second_innings['batting_team'] == second_innings['winner']).astype(int)

print("Match state features calculated")
print(second_innings[['match_id', 'over', 'cumulative_runs', 'cumulative_wickets', 
                        'runs_needed', 'current_rr', 'required_rr', 'batting_team_won']].head(10))

In [ ]:
# Take snapshot at end of each over
over_snapshots = second_innings.groupby(['match_id', 'over']).last().reset_index()

# Filter valid rows
over_snapshots = over_snapshots[
    (over_snapshots['balls_remaining'] >= 0) &
    (over_snapshots['runs_needed'] > 0) &
    (over_snapshots['over'] >= 1)
]

print(f"Total over snapshots: {len(over_snapshots)}")

# Select features for model
features = ['over', 'cumulative_runs', 'cumulative_wickets', 
            'runs_needed', 'current_rr', 'required_rr', 'balls_remaining']
target = 'batting_team_won'

X = over_snapshots[features].dropna()
y = over_snapshots.loc[X.index, target]

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'accuracy': acc, 'auc': auc, 'model': model, 'proba': y_prob}

    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  AUC-ROC:  {auc:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))

for name, result in results.items():
    if name == 'Logistic Regression':
        y_prob = result['proba']
    else:
        y_prob = result['proba']

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {result['auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison — Win Probability Models', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('charts/22_roc_curves.png', dpi=150)
plt.show()

In [ ]:
best_model = results['Random Forest']['model']

importance_df = pd.DataFrame({
    'feature': features,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance — Random Forest Win Probability Model', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('charts/23_feature_importance.png', dpi=150)
plt.show()

print(importance_df)

In [ ]:
def predict_win_probability(over, runs_scored, wickets_lost, 
                             target, model, scaler_obj=None):
    balls_bowled = over * 6
    balls_remaining = 120 - balls_bowled
    runs_needed = target - runs_scored + 1
    current_rr = (runs_scored / balls_bowled) * 6 if balls_bowled > 0 else 0
    required_rr = (runs_needed / balls_remaining) * 6 if balls_remaining > 0 else 99

    features_input = np.array([[over, runs_scored, wickets_lost,
                                  runs_needed, current_rr, required_rr, balls_remaining]])

    if scaler_obj:
        features_input = scaler_obj.transform(features_input)

    prob = model.predict_proba(features_input)[0][1]
    return prob

# Test it with a realistic scenario
rf_model = results['Random Forest']['model']

print("=== WIN PROBABILITY CALCULATOR ===")
print()

scenarios = [
    (5,  45, 0, 160, "Early game — decent start"),
    (10, 80, 2, 160, "Midway — on track"),
    (15, 110, 4, 160, "15th over — under pressure"),
    (18, 145, 5, 160, "Death overs — tight finish"),
    (10, 60, 5, 160, "Midway — in trouble"),
]

for over, runs, wkts, target, description in scenarios:
    prob = predict_win_probability(over, runs, wkts, target, rf_model)
    print(f"Over {over:2d} | Runs: {runs:3d} | Wickets: {wkts} | Target: {target}")
    print(f"Description: {description}")
    print(f"Win Probability: {prob*100:.1f}%")
    print()

In [ ]:
# Simulate a match progression
target = 165
overs = list(range(1, 20))

scenario_ahead = []
scenario_behind = []
scenario_tight = []

for over in overs:
    balls = over * 6
    # Scenario 1: Chasing comfortably
    runs_ahead = int(over * 9.5)
    p1 = predict_win_probability(over, runs_ahead, 1, target, rf_model)
    scenario_ahead.append(p1)

    # Scenario 2: Behind the rate
    runs_behind = int(over * 6.5)
    p2 = predict_win_probability(over, runs_behind, 3, target, rf_model)
    scenario_behind.append(p2)

    # Scenario 3: Neck and neck
    runs_tight = int(over * 8.0)
    p3 = predict_win_probability(over, runs_tight, 2, target, rf_model)
    scenario_tight.append(p3)

plt.figure(figsize=(14, 7))
plt.plot(overs, [p*100 for p in scenario_ahead], 'g-o', linewidth=2, label='Chasing Comfortably (~9.5 RPO)', markersize=6)
plt.plot(overs, [p*100 for p in scenario_tight], 'y-s', linewidth=2, label='Neck and Neck (~8 RPO)', markersize=6)
plt.plot(overs, [p*100 for p in scenario_behind], 'r-^', linewidth=2, label='Behind the Rate (~6.5 RPO)', markersize=6)

plt.axhline(y=50, color='white', linestyle='--', alpha=0.3, label='50% threshold')
plt.xlabel('Over Number')
plt.ylabel('Win Probability (%)')
plt.title(f'Win Probability Progression — Chasing {target} Runs', fontsize=14, fontweight='bold')
plt.legend()
plt.ylim(0, 100)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('charts/24_win_probability_scenarios.png', dpi=150)
plt.show()